# 01 — Dataset definitions for 2.5D vertebra / lumbar-level heatmap localization

This notebook contains only reusable dataset/preprocessing definitions.

Dataset variants are intentionally limited to:

1. `no_aug` — resize + normalization only.
2. `intensity_aug` — same intensity augmentation applied consistently to all channels in the 2.5D stack.

No crop, rotation, translation, perspective transform, or geometric augmentation is used.

In [ ]:
import os
import random
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset

try:
    import pydicom
except ImportError as e:
    raise ImportError("Please install pydicom first: pip install pydicom") from e

LEVEL_TO_IDX = {level: i for i, level in enumerate(LEVELS)}
IDX_TO_LEVEL = {i: level for level, i in LEVEL_TO_IDX.items()}

SEVERITY_TO_COLOR = {
    "Normal/Mild": "tab:green",
    "Moderate": "gold",
    "Severe": "tab:red",
}

In [ ]:
@dataclass
class LocalizationDatasetConfig:
    img_size: int = 256
    num_slices: int = 3
    sigma: float = 5.0

    # Image normalization before augmentation.
    normalize_mode: str = "percentile"  # "percentile" or "minmax"
    p_low: float = 1.0
    p_high: float = 99.5

    # Intensity-only augmentation. No geometry is changed.
    use_intensity_aug: bool = False
    aug_prob: float = 0.85
    contrast_range: Tuple[float, float] = (0.70, 1.50)
    brightness_range: Tuple[float, float] = (-0.12, 0.12)
    gamma_range: Tuple[float, float] = (0.70, 1.50)
    gaussian_blur_prob: float = 0.30
    gaussian_blur_sigma_range: Tuple[float, float] = (0.20, 1.20)
    gaussian_noise_prob: float = 0.20
    gaussian_noise_std_range: Tuple[float, float] = (0.005, 0.030)

    seed: int = 42

    def __post_init__(self):
        assert self.num_slices % 2 == 1, "num_slices should be odd, e.g. 3 or 5."
        assert self.img_size > 0
        assert self.sigma > 0

In [ ]:
def filter_localization_rows(
    df: pd.DataFrame,
    series_descriptions: Optional[Sequence[str]] = ("Sagittal T2/STIR",),
    conditions: Optional[Sequence[str]] = ("Spinal Canal Stenosis",),
    drop_missing_files: bool = True,
) -> pd.DataFrame:
    """Filter rows for a localization experiment."""
    out = df.copy()
    if "level_idx" not in out.columns:
        out["level_idx"] = out["level"].map(LEVEL_TO_IDX)

    if series_descriptions is not None:
        out = out[out["series_description"].isin(series_descriptions)]
    if conditions is not None:
        out = out[out["condition"].isin(conditions)]

    out = out.dropna(subset=["x", "y", "level_idx", "img_path"]).copy()

    if drop_missing_files:
        exists = out["img_path"].map(lambda p: Path(p).exists())
        missing_count = int((~exists).sum())
        if missing_count:
            print(f"Dropping {missing_count} rows with missing DICOM files.")
        out = out[exists]

    out["level_idx"] = out["level_idx"].astype(int)
    return out.reset_index(drop=True)


def split_by_study(
    df: pd.DataFrame,
    val_frac: float = 0.15,
    test_frac: float = 0.15,
    seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Study-level split to avoid leakage across slices from the same patient/study."""
    rng = np.random.default_rng(seed)
    studies = np.array(sorted(df["study_id"].unique()))
    rng.shuffle(studies)

    n = len(studies)
    n_test = int(round(n * test_frac))
    n_val = int(round(n * val_frac))

    test_ids = set(studies[:n_test])
    val_ids = set(studies[n_test:n_test + n_val])
    train_ids = set(studies[n_test + n_val:])

    train_df = df[df["study_id"].isin(train_ids)].reset_index(drop=True)
    val_df = df[df["study_id"].isin(val_ids)].reset_index(drop=True)
    test_df = df[df["study_id"].isin(test_ids)].reset_index(drop=True)
    return train_df, val_df, test_df

In [ ]:
def read_dicom_array(path: Union[str, Path]) -> np.ndarray:
    ds = pydicom.dcmread(str(path))
    arr = ds.pixel_array.astype(np.float32)

    # Respect MONOCHROME1, where high values are dark.
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def normalize_image(
    arr: np.ndarray,
    mode: str = "percentile",
    p_low: float = 1.0,
    p_high: float = 99.5,
    eps: float = 1e-6,
) -> np.ndarray:
    arr = arr.astype(np.float32)
    if mode == "percentile":
        lo, hi = np.percentile(arr, [p_low, p_high])
    elif mode == "minmax":
        lo, hi = float(arr.min()), float(arr.max())
    else:
        raise ValueError(f"Unknown normalize mode: {mode}")
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / max(hi - lo, eps)
    return arr.astype(np.float32)


def list_series_dicoms(img_path: Union[str, Path]) -> List[Path]:
    img_path = Path(img_path)
    folder = img_path.parent
    return sorted(folder.glob("*.dcm"), key=lambda p: int(p.stem) if p.stem.isdigit() else p.name)


def load_25d_stack(
    img_path: Union[str, Path],
    num_slices: int = 3,
    normalize_mode: str = "percentile",
    p_low: float = 1.0,
    p_high: float = 99.5,
) -> np.ndarray:
    """Load neighboring slices as CxHxW float32 in [0, 1].

    For edge slices, the nearest available slice is repeated.

    Important:
    Some Axial T2 series can contain neighboring DICOM slices with different
    pixel-array shapes. Therefore, all neighboring slices are resized to the
    center slice shape before np.stack().
    """
    files = list_series_dicoms(img_path)
    img_path = Path(img_path)

    if not files:
        raise FileNotFoundError(f"No DICOM files found in {img_path.parent}")

    try:
        center_idx = files.index(img_path)
    except ValueError:
        stems = [p.stem for p in files]
        center_idx = stems.index(img_path.stem)

    # Use center slice as the reference shape.
    center_arr_raw = read_dicom_array(files[center_idx])
    ref_h, ref_w = center_arr_raw.shape[:2]

    radius = num_slices // 2
    channels = []

    for offset in range(-radius, radius + 1):
        idx = int(np.clip(center_idx + offset, 0, len(files) - 1))

        arr = read_dicom_array(files[idx])
        arr = normalize_image(
            arr,
            mode=normalize_mode,
            p_low=p_low,
            p_high=p_high,
        )

        # Fix for Axial T2: force every neighboring slice to center-slice shape.
        if arr.shape[:2] != (ref_h, ref_w):
            arr = cv2.resize(
                arr,
                (ref_w, ref_h),
                interpolation=cv2.INTER_AREA,
            ).astype(np.float32)

        channels.append(arr)

    return np.stack(channels, axis=0).astype(np.float32)

In [ ]:
def resize_stack_and_point(
    stack: np.ndarray,
    x: float,
    y: float,
    out_size: int,
) -> Tuple[np.ndarray, float, float]:
    """Resize CxHxW stack and transform one point. This is not augmentation."""
    c, h, w = stack.shape
    resized = np.stack([
        cv2.resize(stack[i], (out_size, out_size), interpolation=cv2.INTER_AREA)
        for i in range(c)
    ], axis=0).astype(np.float32)

    x_new = float(x) * out_size / float(w)
    y_new = float(y) * out_size / float(h)
    return resized, x_new, y_new


def apply_intensity_augmentation_same_for_stack(
    stack: np.ndarray,
    cfg: LocalizationDatasetConfig,
    rng: np.random.Generator,
) -> np.ndarray:
    """Apply intensity augmentation consistently to all channels of a CxHxW stack.

    Important: the same brightness, contrast, gamma, blur sigma and noise map are used for every
    slice/channel. There is no crop, rotation, translation, scaling or perspective transform.
    """
    if not cfg.use_intensity_aug or rng.random() > cfg.aug_prob:
        return stack.astype(np.float32)

    out = stack.astype(np.float32).copy()

    contrast = rng.uniform(*cfg.contrast_range)
    brightness = rng.uniform(*cfg.brightness_range)
    gamma = rng.uniform(*cfg.gamma_range)

    out = out * float(contrast) + float(brightness)
    out = np.clip(out, 0.0, 1.0)
    out = np.power(out, float(gamma))

    if cfg.gaussian_blur_prob > 0 and rng.random() < cfg.gaussian_blur_prob:
        sigma = float(rng.uniform(*cfg.gaussian_blur_sigma_range))
        out = np.stack([
            cv2.GaussianBlur(out[i], ksize=(0, 0), sigmaX=sigma, sigmaY=sigma)
            for i in range(out.shape[0])
        ], axis=0).astype(np.float32)

    if cfg.gaussian_noise_prob > 0 and rng.random() < cfg.gaussian_noise_prob:
        std = float(rng.uniform(*cfg.gaussian_noise_std_range))
        # Same noise pattern for every channel to avoid slice-specific artificial differences.
        noise_2d = rng.normal(0.0, std, size=out.shape[1:]).astype(np.float32)
        out = out + noise_2d[None, :, :]

    return np.clip(out, 0.0, 1.0).astype(np.float32)

In [ ]:
def make_gaussian_heatmap(
    x: float,
    y: float,
    size: int,
    sigma: float,
    normalize_peak: bool = True,
) -> np.ndarray:
    yy, xx = np.meshgrid(
        np.arange(size, dtype=np.float32),
        np.arange(size, dtype=np.float32),
        indexing="ij",
    )
    heatmap = np.exp(-((xx - float(x)) ** 2 + (yy - float(y)) ** 2) / (2.0 * float(sigma) ** 2))
    if normalize_peak:
        heatmap = heatmap / max(float(heatmap.max()), 1e-8)
    return heatmap.astype(np.float32)


def heatmap_argmax_xy(heatmap: Union[np.ndarray, torch.Tensor]) -> Tuple[float, float]:
    if torch.is_tensor(heatmap):
        heatmap = heatmap.detach().cpu().numpy()
    idx = int(np.argmax(heatmap))
    y, x = np.unravel_index(idx, heatmap.shape)
    return float(x), float(y)


def soft_argmax_2d(prob: torch.Tensor, beta: float = 50.0) -> torch.Tensor:
    """Differentiable coordinate extraction from Bx1xHxW heatmaps."""
    b, c, h, w = prob.shape
    assert c == 1
    flat = prob.view(b, -1)
    weights = torch.softmax(flat * beta, dim=1)
    yy, xx = torch.meshgrid(
        torch.arange(h, device=prob.device, dtype=prob.dtype),
        torch.arange(w, device=prob.device, dtype=prob.dtype),
        indexing="ij",
    )
    xx = xx.reshape(-1)
    yy = yy.reshape(-1)
    x = (weights * xx[None, :]).sum(dim=1)
    y = (weights * yy[None, :]).sum(dim=1)
    return torch.stack([x, y], dim=1)

In [ ]:
class VertebraHeatmap25DDataset(Dataset):
    """2.5D heatmap dataset with target lumbar level as model metadata.

    Each sample returns:
    - image: CxHxW tensor, usually 3xHxW
    - target: 1xHxW Gaussian heatmap
    - point_xy: scaled target coordinate in resized image space
    - level_idx: integer target level metadata
    """
    def __init__(
        self,
        df: pd.DataFrame,
        config: LocalizationDatasetConfig,
        is_train: bool = True,
    ):
        self.df = df.reset_index(drop=True).copy()
        if "level_idx" not in self.df.columns:
            self.df["level_idx"] = self.df["level"].map(LEVEL_TO_IDX)
        self.cfg = config
        self.is_train = bool(is_train)
        self.rng = np.random.default_rng(config.seed if is_train else config.seed + 10_000)

    def __len__(self):
        return len(self.df)

    def set_sigma(self, sigma: float):
        self.cfg.sigma = float(sigma)

    def __getitem__(self, idx: int) -> Dict[str, Union[torch.Tensor, str, int, float]]:
        row = self.df.iloc[idx]
        x = float(row["x"])
        y = float(row["y"])

        stack = load_25d_stack(
            row["img_path"],
            num_slices=self.cfg.num_slices,
            normalize_mode=self.cfg.normalize_mode,
            p_low=self.cfg.p_low,
            p_high=self.cfg.p_high,
        )

        stack, x, y = resize_stack_and_point(stack, x, y, self.cfg.img_size)

        if self.is_train:
            stack = apply_intensity_augmentation_same_for_stack(stack, self.cfg, self.rng)

        heatmap = make_gaussian_heatmap(x, y, self.cfg.img_size, self.cfg.sigma)

        return {
            "image": torch.from_numpy(stack).float(),
            "target": torch.from_numpy(heatmap[None, :, :]).float(),
            "point_xy": torch.tensor([x, y], dtype=torch.float32),
            "level_idx": torch.tensor(int(row["level_idx"]), dtype=torch.long),
            "level": str(row["level"]),
            "condition": str(row.get("condition", "")),
            "severity": str(row.get("severity", "")),
            "study_id": int(row["study_id"]),
            "series_id": int(row["series_id"]),
            "instance_number": int(row["instance_number"]),
            "img_path": str(row["img_path"]),
        }

In [ ]:
def make_dataset_variants(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    base_config: LocalizationDatasetConfig,
) -> Dict[str, Dict[str, VertebraHeatmap25DDataset]]:
    """Create the only two supported dataset variants: no_aug and intensity_aug."""
    import copy

    variants = {}

    cfg_no_aug = copy.deepcopy(base_config)
    cfg_no_aug.use_intensity_aug = False
    variants["no_aug"] = {
        "train": VertebraHeatmap25DDataset(train_df, cfg_no_aug, is_train=True),
        "val": VertebraHeatmap25DDataset(val_df, cfg_no_aug, is_train=False),
        "test": VertebraHeatmap25DDataset(test_df, cfg_no_aug, is_train=False),
    }

    cfg_intensity = copy.deepcopy(base_config)
    cfg_intensity.use_intensity_aug = True
    variants["intensity_aug"] = {
        "train": VertebraHeatmap25DDataset(train_df, cfg_intensity, is_train=True),
        "val": VertebraHeatmap25DDataset(val_df, cfg_intensity, is_train=False),
        "test": VertebraHeatmap25DDataset(test_df, cfg_intensity, is_train=False),
    }

    return variants

In [ ]:
def plot_localization_sample(sample, pred: Optional[Union[np.ndarray, torch.Tensor]] = None, title: str = ""):
    image = sample["image"]
    target = sample["target"]
    if torch.is_tensor(image):
        image_np = image.detach().cpu().numpy()
    else:
        image_np = np.asarray(image)
    if torch.is_tensor(target):
        target_np = target.detach().cpu().numpy()[0]
    else:
        target_np = np.asarray(target)[0]

    center = image_np[image_np.shape[0] // 2]
    tx, ty = sample["point_xy"].detach().cpu().numpy() if torch.is_tensor(sample["point_xy"]) else sample["point_xy"]

    ncols = 3 if pred is not None else 2
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))
    if ncols == 2:
        axes = list(axes)

    axes[0].imshow(center, cmap="gray")
    axes[0].scatter([tx], [ty], c="lime", s=35)
    axes[0].set_title("center slice + target")
    axes[0].axis("off")

    axes[1].imshow(center, cmap="gray")
    axes[1].imshow(target_np, cmap="hot", alpha=0.45)
    axes[1].set_title("target heatmap")
    axes[1].axis("off")

    if pred is not None:
        if torch.is_tensor(pred):
            pred_np = pred.detach().cpu().numpy()
        else:
            pred_np = np.asarray(pred)
        px, py = heatmap_argmax_xy(pred_np)
        axes[2].imshow(center, cmap="gray")
        axes[2].imshow(pred_np, cmap="hot", alpha=0.45)
        axes[2].scatter([px], [py], c="cyan", s=35)
        axes[2].set_title("prediction")
        axes[2].axis("off")

    full_title = title or f"level={sample['level']} | severity={sample['severity']}"
    fig.suptitle(full_title)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 3b. External coordinate adapter and replicated-slice dataset
# ============================================================

def _first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def _resolve_external_path(path_value, image_root=EXTERNAL_IMAGE_ROOT):
    p = Path(str(path_value))
    if p.is_absolute():
        return str(p)
    return str(Path(image_root) / p)


def _load_grayscale_image_for_external(path):
    """Load PNG/JPG/TIFF with PIL; fall back to DICOM via pydicom when needed."""
    path = str(path)
    try:
        with Image.open(path) as im:
            return ImageOps.grayscale(im)
    except Exception as pil_error:
        suffix = Path(path).suffix.lower()
        if suffix in {".dcm", ".dicom", ""}:
            try:
                import pydicom
                ds = pydicom.dcmread(path)
                arr = ds.pixel_array.astype(np.float32)
                arr -= np.nanmin(arr)
                mx = np.nanmax(arr)
                if mx > 0:
                    arr /= mx
                arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
                return Image.fromarray(arr, mode="L")
            except Exception as dicom_error:
                raise RuntimeError(
                    f"Could not load external image as PIL or DICOM: {path}\n"
                    f"PIL error: {pil_error}\nDICOM error: {dicom_error}"
                )
        raise RuntimeError(f"Could not load external image with PIL: {path}\nError: {pil_error}")


def build_external_localization_df(
    csv_path=EXTERNAL_COORDS_CSV,
    image_root=EXTERNAL_IMAGE_ROOT,
    source_name=EXTERNAL_SOURCE_NAME,
    levels=LEVELS,
    compute_image_size=True,
):
    """
    Convert df_coords.csv from Localization_lumbar_spine_aug into a data_merged-like
    long dataframe suitable for the conditional localization training workflow.

    Required after standardization:
      img_path, x_norm, y_norm, level
    """
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f"External coordinate CSV not found, skipping: {csv_path}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path).copy()

    if "exists" in df.columns:
        df = df[df["exists"].astype(bool)].copy()

    path_col = _first_existing(df.columns, ["full_path", "path", "image_path", "filepath", "file_path", "filename", "img_path"])
    if path_col is None:
        raise ValueError(f"External CSV has no path column. Available columns: {list(df.columns)}")

    if {"relative_x", "relative_y"}.issubset(df.columns):
        x_norm = df["relative_x"].astype(float)
        y_norm = df["relative_y"].astype(float)
    elif {"x_norm", "y_norm"}.issubset(df.columns):
        x_norm = df["x_norm"].astype(float)
        y_norm = df["y_norm"].astype(float)
    else:
        x_col = _first_existing(df.columns, ["x", "coord_x", "center_x", "px", "column"])
        y_col = _first_existing(df.columns, ["y", "coord_y", "center_y", "py", "row"])
        w_col = _first_existing(df.columns, ["image_width", "width", "img_width", "W"])
        h_col = _first_existing(df.columns, ["image_height", "height", "img_height", "H"])
        if x_col is None or y_col is None:
            raise ValueError("External CSV needs relative_x/relative_y, x_norm/y_norm, or x/y-like columns.")
        if w_col is None or h_col is None:
            raise ValueError("Absolute external coordinates require image_width/image_height columns.")
        x_norm = df[x_col].astype(float) / np.maximum(df[w_col].astype(float) - 1, 1)
        y_norm = df[y_col].astype(float) / np.maximum(df[h_col].astype(float) - 1, 1)

    level_col = _first_existing(df.columns, ["level", "vertebra", "disc_level"])
    if level_col is None:
        raise ValueError(f"External CSV has no level column. Available columns: {list(df.columns)}")

    out = pd.DataFrame({
        "img_path": df[path_col].apply(lambda p: _resolve_external_path(p, image_root=image_root)),
        "level": df[level_col].astype(str),
        "x_norm": np.clip(x_norm.astype(float), 0.0, 1.0),
        "y_norm": np.clip(y_norm.astype(float), 0.0, 1.0),
    })

    out = out[out["level"].isin(levels)].copy()
    out = out.dropna(subset=["img_path", "level", "x_norm", "y_norm"]).reset_index(drop=True)

    # Use path-derived IDs because the external CSV may have a repeated/non-patient study_id.
    out["external_image_id"] = out["img_path"].apply(lambda p: Path(p).stem)
    if "study_id" in df.columns:
        raw_study = df.loc[out.index, "study_id"].astype(str).values if len(out) <= len(df) else out["external_image_id"].values
        out["original_study_id"] = raw_study
    out["study_id"] = source_name + "_" + out["external_image_id"].astype(str)
    out["series_id"] = source_name + "_single_slice"
    out["instance_number"] = 0
    out["condition"] = "Spinal Canal Stenosis"
    out["series_description"] = "External single-slice lumbar localization"
    out["severity"] = "external_unlabeled"
    out["source"] = source_name
    out["level_idx"] = out["level"].map({lvl: i for i, lvl in enumerate(levels)}).astype(int)
    out["row_id"] = (
        out["source"].astype(str) + "_" +
        out["external_image_id"].astype(str) + "_" +
        out["level"].str.lower().str.replace("/", "_", regex=False)
    )

    if compute_image_size:
        size_cache = {}
        for p in out["img_path"].drop_duplicates():
            try:
                im = _load_grayscale_image_for_external(p)
                size_cache[p] = im.size  # (W, H)
            except Exception as e:
                print(f"Warning: could not read external image size for {p}: {e}")
                size_cache[p] = (np.nan, np.nan)
        out["image_width"] = out["img_path"].map(lambda p: size_cache[p][0])
        out["image_height"] = out["img_path"].map(lambda p: size_cache[p][1])
    else:
        out["image_width"] = np.nan
        out["image_height"] = np.nan

    out["x"] = out["x_norm"] * (out["image_width"].fillna(IMG_SIZE) - 1)
    out["y"] = out["y_norm"] * (out["image_height"].fillna(IMG_SIZE) - 1)

    keep_cols = [
        "row_id", "study_id", "series_id", "condition", "level", "series_description",
        "instance_number", "x", "y", "severity", "img_path", "image_height", "image_width",
        "x_norm", "y_norm", "source", "level_idx", "external_image_id",
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[keep_cols].reset_index(drop=True)

    # Critical sanity checks.
    print(f"External localization rows: {len(out):,}")
    print(f"External unique images: {out['img_path'].nunique():,}")
    print("External level counts:")
    display(out["level"].value_counts().reindex(levels))

    per_image_levels = out.groupby("img_path")["level"].nunique()
    print("External number of annotated levels per image:")
    display(per_image_levels.value_counts().sort_index())

    missing_files = [p for p in out["img_path"].drop_duplicates().head(20) if not Path(p).exists()]
    if missing_files:
        print("Warning: some external image paths do not exist. First missing paths:")
        for p in missing_files[:5]:
            print("  ", p)

    return out


def make_single_point_heatmap(img_size, x, y, sigma):
    yy = np.arange(img_size, dtype=np.float32)[:, None]
    xx = np.arange(img_size, dtype=np.float32)[None, :]
    heatmap = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2.0 * float(sigma) ** 2))
    return heatmap.astype(np.float32)


class ExternalReplicatedSliceLocalizationDataset(Dataset):
    """
    Dataset for external single-slice lumbar coordinate rows.

    Returns the same keys used by the localization training loop:
      image:    FloatTensor [NUM_SLICES, IMG_SIZE, IMG_SIZE]
      target:   FloatTensor [1, IMG_SIZE, IMG_SIZE]
      point_xy: FloatTensor [2]
      level_idx LongTensor scalar

    The image is a single grayscale PNG/JPG/DICOM slice replicated over all channels.
    """
    def __init__(self, df, config, levels=LEVELS, use_intensity_aug=False):
        self.df = df.reset_index(drop=True).copy()
        self.config = config
        # Expose the same attribute name as the RSNA localization dataset.
        # The training loop reads train_loader.dataset.cfg.sigma.
        self.cfg = config
        self.levels = list(levels)
        self.level_to_idx = {lvl: i for i, lvl in enumerate(self.levels)}
        self.img_size = int(config.img_size)
        self.num_slices = int(config.num_slices)
        self.sigma = float(config.sigma)
        self.use_intensity_aug = bool(use_intensity_aug)

        if "level_idx" not in self.df.columns:
            self.df["level_idx"] = self.df["level"].map(self.level_to_idx).astype(int)

    def set_sigma(self, sigma):
        self.sigma = float(sigma)
        if hasattr(self, "cfg"):
            try:
                self.cfg.sigma = float(sigma)
            except Exception:
                pass

    def __len__(self):
        return len(self.df)

    def _load_image_tensor(self, path):
        im = _load_grayscale_image_for_external(path)
        im = im.resize((self.img_size, self.img_size), resample=Image.BILINEAR)
        arr = np.asarray(im, dtype=np.float32) / 255.0

        if self.use_intensity_aug:
            # Conservative intensity-only augmentation; no spatial transforms here because
            # these samples are already used as an external extension source.
            if random.random() < 0.50:
                arr = np.clip(arr * random.uniform(0.85, 1.15), 0.0, 1.0)
            if random.random() < 0.25:
                arr = np.clip(arr + np.random.normal(0.0, 0.015, size=arr.shape).astype(np.float32), 0.0, 1.0)

        # Match the common MRI normalization convention used in the pretraining notebook.
        arr = (arr - 0.5) / 0.5
        image_1 = torch.from_numpy(arr).float().unsqueeze(0)  # [1,H,W]
        return image_1.repeat(self.num_slices, 1, 1)          # [C,H,W]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = float(np.clip(row["x_norm"], 0.0, 1.0)) * (self.img_size - 1)
        y = float(np.clip(row["y_norm"], 0.0, 1.0)) * (self.img_size - 1)
        level_idx = int(row["level_idx"])

        image = self._load_image_tensor(row["img_path"])
        target = make_single_point_heatmap(self.img_size, x, y, self.sigma)

        return {
            "image": image,
            "target": torch.from_numpy(target).float().unsqueeze(0),
            "point_xy": torch.tensor([x, y], dtype=torch.float32),
            "level_idx": torch.tensor(level_idx, dtype=torch.long),
        }


class MixedLocalizationDataset(Dataset):
    """Concatenate datasets while preserving .cfg, combined .df, and sigma propagation."""
    def __init__(self, datasets):
        self.datasets = [d for d in datasets if d is not None and len(d) > 0]
        self.lengths = [len(d) for d in self.datasets]
        self.cum_lengths = np.cumsum(self.lengths).tolist()

        # Mirror the original RSNA dataset API. The existing training loop expects
        # train_loader.dataset.cfg.sigma, so the mixed wrapper must expose cfg.
        self.cfg = None
        for d in self.datasets:
            if hasattr(d, "cfg"):
                self.cfg = d.cfg
                break
            if hasattr(d, "config"):
                self.cfg = d.config
                break

        dfs = []
        for d in self.datasets:
            if hasattr(d, "df"):
                dfs.append(d.df.copy())
        self.df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    def __len__(self):
        return int(sum(self.lengths))

    def set_sigma(self, sigma):
        sigma = float(sigma)
        if self.cfg is not None:
            try:
                self.cfg.sigma = sigma
            except Exception:
                pass
        for d in self.datasets:
            if hasattr(d, "set_sigma"):
                d.set_sigma(sigma)
            elif hasattr(d, "sigma"):
                d.sigma = sigma

    def __getitem__(self, idx):
        if idx < 0:
            idx = len(self) + idx
        ds_start = 0
        for ds, end in zip(self.datasets, self.cum_lengths):
            if idx < end:
                sample = ds[idx - ds_start]
                # The training loop only needs these keys. Keeping a strict common
                # schema avoids PyTorch default-collate failures when RSNA and
                # external datasets expose different metadata keys.
                return {
                    "image": sample["image"],
                    "target": sample["target"],
                    "point_xy": sample["point_xy"],
                    "level_idx": sample["level_idx"],
                }
            ds_start = end
        raise IndexError(idx)

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

def build_relative_coordinate_prior_maps_from_norm_cols(
    train_df,
    img_size=IMG_SIZE,
    sigma=PRIOR_SIGMA,
    levels=LEVELS,
    eps=PRIOR_EPS,
):
    """
    Build one prior map per level from already existing normalized coordinates:
      - x_norm
      - y_norm

    Expected columns in train_df:
      - level
      - optionally level_idx
      - x_norm
      - y_norm
    """

    priors = np.zeros((len(levels), img_size, img_size), dtype=np.float32)
    counts = np.zeros(len(levels), dtype=np.int64)

    for _, row in train_df.iterrows():
        if pd.isna(row["x_norm"]) or pd.isna(row["y_norm"]):
            continue

        if "level_idx" in train_df.columns and not pd.isna(row["level_idx"]):
            level_idx = int(row["level_idx"])
        else:
            level_idx = levels.index(str(row["level"]))

        x_rel = float(row["x_norm"])
        y_rel = float(row["y_norm"])

        x_rel = np.clip(x_rel, 0.0, 1.0)
        y_rel = np.clip(y_rel, 0.0, 1.0)

        xi = int(round(x_rel * (img_size - 1)))
        yi = int(round(y_rel * (img_size - 1)))

        priors[level_idx, yi, xi] += 1.0
        counts[level_idx] += 1

    for level_idx in range(len(levels)):
        if counts[level_idx] == 0:
            priors[level_idx] = np.ones((img_size, img_size), dtype=np.float32)
        else:
            priors[level_idx] = gaussian_filter(
                priors[level_idx],
                sigma=sigma,
                mode="constant",
            )

            mx = priors[level_idx].max()
            if mx > eps:
                priors[level_idx] /= mx

            priors[level_idx] = np.clip(priors[level_idx], eps, 1.0)

    print("Built relative-coordinate priors:", priors.shape)
    print("Counts per level:", dict(zip(levels, counts.tolist())))

    return priors